In [1]:
# ============================================
# CONFIGURACIÓN INICIAL Y LIBRERÍAS
# ============================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Procesamiento de señales
from scipy import signal
from scipy.stats import skew, kurtosis, entropy, pearsonr
from scipy.integrate import trapz

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, KFold, LeaveOneOut
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, accuracy_score
from xgboost import XGBRegressor, XGBClassifier
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.impute import SimpleImputer


ImportError: cannot import name 'trapz' from 'scipy.integrate' (/usr/local/lib/python3.12/dist-packages/scipy/integrate/__init__.py)

In [ ]:
# Configuración de rutas
BASE_PATH = Path("/ruta/a/tus/datos")  # Ajusta esta ruta
FNIRS_PATH = BASE_PATH / "fNIRS_data"
TRIGGERS_FILE = BASE_PATH / "fNIRS_data/3F_Triggers.csv"  # Ajusta según el sujeto

In [ ]:
# ============================================
# 1. CARGA DE DATOS FNIRS
# ============================================

def load_fnirs_data(subject_id, data_type):
    """
    Carga datos fNIRS para un sujeto específico
    data_type: 'ohb', 'dohb', 'tothb', 'O2'
    """
    # Buscar archivo que coincida con el patrón
    pattern = f"{subject_id}_*{data_type}.csv"
    files = list(FNIRS_PATH.glob(pattern))

    if not files:
        print(f"No se encontró archivo para {subject_id} - {data_type}")
        return None

    filepath = files[0]
    print(f"Cargando {filepath.name}")

    # Leer datos (asumiendo que son valores separados por coma)
    data = pd.read_csv(filepath, header=None)

    # Si es una sola columna, tomar todos los valores
    if data.shape[1] == 1:
        values = data.iloc[:, 0].values
    else:
        # Si hay múltiples columnas (canales), tomar la media
        values = data.mean(axis=1).values

    return values

def load_all_fnirs_for_subject(subject_id):
    """
    Carga todos los tipos de datos fNIRS para un sujeto
    """
    data_types = ['ohb', 'dohb', 'tothb', 'O2']
    subject_data = {}

    for data_type in data_types:
        values = load_fnirs_data(subject_id, data_type)
        if values is not None:
            subject_data[data_type] = values

    return subject_data

In [ ]:
# ============================================
# 2. CARGA DE TRIGGERS
# ============================================

def load_triggers(subject_id):
    """
    Carga los triggers para un sujeto
    """
    pattern = f"{subject_id}_*Triggers.csv"
    files = list(FNIRS_PATH.glob(pattern))

    if not files:
        print(f"No se encontraron triggers para {subject_id}")
        return None

    filepath = files[0]
    print(f"Cargando triggers: {filepath.name}")

    # Leer el archivo de triggers
    df = pd.read_csv(filepath, header=None)

    # La estructura parece ser: primera fila nombres, segunda fila valores
    if len(df) >= 2:
        trigger_names = df.iloc[0, :].values
        trigger_times = df.iloc[1, :].values.astype(float)

        triggers = {}
        for name, time in zip(trigger_names, trigger_times):
            triggers[name] = time

        return triggers

    return None

In [ ]:
# ============================================
# 3. EXTRACCIÓN DE CARACTERÍSTICAS DE FNIRS
# ============================================

def extract_fnirs_features(fnirs_data, triggers, fs=7.6294):
    """
    Extrae características de las señales fNIRS
    """
    features = {}

    if not fnirs_data:
        return features

    # Para cada tipo de señal
    for signal_type, values in fnirs_data.items():
        if values is None or len(values) < fs * 10:  # Mínimo 10 segundos
            continue

        prefix = f"fnirs_{signal_type}"

        # Características básicas
        features[f'{prefix}_mean'] = np.mean(values)
        features[f'{prefix}_std'] = np.std(values)
        features[f'{prefix}_skew'] = skew(values)
        features[f'{prefix}_kurtosis'] = kurtosis(values)
        features[f'{prefix}_min'] = np.min(values)
        features[f'{prefix}_max'] = np.max(values)
        features[f'{prefix}_range'] = np.max(values) - np.min(values)
        features[f'{prefix}_slope'] = np.polyfit(range(len(values)), values, 1)[0]
        features[f'{prefix}_auc'] = trapz(values) / len(values)

        # Características espectrales
        if len(values) > fs * 2:
            freqs, psd = signal.welch(values, fs, nperseg=min(128, len(values)))
            features[f'{prefix}_power_total'] = np.sum(psd)
            if len(freqs) > 0:
                features[f'{prefix}_peak_freq'] = freqs[np.argmax(psd)]

            # Bandas de frecuencia típicas para fNIRS
            # Las señales hemodinámicas son de baja frecuencia
            vlf_mask = freqs < 0.04
            lf_mask = (freqs >= 0.04) & (freqs < 0.15)
            hf_mask = freqs >= 0.15

            features[f'{prefix}_power_vlf'] = np.sum(psd[vlf_mask]) if any(vlf_mask) else 0
            features[f'{prefix}_power_lf'] = np.sum(psd[lf_mask]) if any(lf_mask) else 0
            features[f'{prefix}_power_hf'] = np.sum(psd[hf_mask]) if any(hf_mask) else 0
            if features[f'{prefix}_power_hf'] > 0:
                features[f'{prefix}_lf_hf_ratio'] = features[f'{prefix}_power_lf'] / features[f'{prefix}_power_hf']

        # Características de la dinámica temporal
        diff_values = np.diff(values)
        features[f'{prefix}_mean_derivative'] = np.mean(diff_values)
        features[f'{prefix}_std_derivative'] = np.std(diff_values)
        features[f'{prefix}_zero_crossings'] = np.sum(np.diff(np.sign(diff_values)) != 0) / len(diff_values)

    # Si hay triggers, extraer características de las condiciones
    if triggers:
        # Por ahora, solo guardamos los tiempos de trigger como características
        for trigger_name, trigger_time in triggers.items():
            # Limpiar nombre del trigger para usarlo como nombre de columna
            clean_name = trigger_name.replace(' ', '_').replace(',', '')
            features[f'trigger_{clean_name}'] = trigger_time

    return features

In [ ]:
# ============================================
# 4. CONSTRUIR DATASET MULTIMODAL COMPLETO
# ============================================

def build_multimodal_dataset(subjects_list):
    """
    Construye dataset completo con características de fNIRS y triggers
    """
    all_features = []

    for subject_id in subjects_list:
        print(f"\n{'='*50}")
        print(f"Procesando sujeto: {subject_id}")
        print('='*50)

        # Cargar datos fNIRS
        fnirs_data = load_all_fnirs_for_subject(subject_id)

        # Cargar triggers
        triggers = load_triggers(subject_id)

        if not fnirs_data:
            print(f"  No hay datos fNIRS para {subject_id}")
            continue

        # Extraer características
        features = {'subject_id': subject_id}

        # Extraer género del subject_id
        if 'F' in subject_id:
            features['gender'] = 'F'
        elif 'M' in subject_id:
            features['gender'] = 'M'

        # Características de fNIRS
        fnirs_features = extract_fnirs_features(fnirs_data, triggers)
        features.update(fnirs_features)

        all_features.append(features)

        # Mostrar resumen
        print(f"  Características extraídas: {len(fnirs_features)}")
        for signal_type in ['ohb', 'dohb', 'tothb', 'O2']:
            if signal_type in fnirs_data:
                print(f"    {signal_type}: {len(fnirs_data[signal_type])} muestras")

    return pd.DataFrame(all_features)

# Lista de sujetos disponibles
subjects = ['3F', '4F', '6M', '8M', '11F']
print("Construyendo dataset multimodal...")
df_multimodal = build_multimodal_dataset(subjects)

print(f"\n{'='*50}")
print("DATASET MULTIMODAL CONSTRUIDO")
print('='*50)
print(f"Dimensiones: {df_multimodal.shape}")
print(f"Sujetos: {df_multimodal['subject_id'].tolist()}")
print(f"Géneros: {df_multimodal['gender'].value_counts().to_dict()}")
print(f"Número de características: {len([c for c in df_multimodal.columns if c not in ['subject_id', 'gender']])}")

In [ ]:
# ============================================
# 5. ANÁLISIS EXPLORATORIO DE DATOS
# ============================================

# Visualizar las señales fNIRS para un sujeto de ejemplo
def plot_fnirs_signals(subject_id, fnirs_data):
    """
    Visualiza las señales fNIRS para un sujeto
    """
    fig, axes = plt.subplots(4, 1, figsize=(15, 12))

    signal_types = ['ohb', 'dohb', 'tothb', 'O2']
    colors = ['green', 'red', 'blue', 'purple']
    titles = ['Oxyhemoglobin (OHb)', 'Deoxyhemoglobin (DOHb)',
              'Total Hemoglobin (tothb)', 'Oxygen Saturation (O2)']

    for i, (signal_type, color, title) in enumerate(zip(signal_types, colors, titles)):
        if signal_type in fnirs_data:
            values = fnirs_data[signal_type]
            time = np.arange(len(values)) / 7.6294  # Convertir a segundos

            axes[i].plot(time, values, color=color, linewidth=1)
            axes[i].set_ylabel('Concentración (mmol)')
            axes[i].set_title(f'{title} - {subject_id}')
            axes[i].grid(True, alpha=0.3)

            if i == 3:
                axes[i].set_xlabel('Tiempo (segundos)')

    plt.tight_layout()
    plt.show()

# Cargar datos completos para visualización
for subject in subjects[:1]:  # Solo el primer sujeto para ejemplo
    fnirs_data = load_all_fnirs_for_subject(subject)
    if fnirs_data:
        plot_fnirs_signals(subject, fnirs_data)

In [ ]:
# ============================================
# 6. ANÁLISIS DE CORRELACIONES ENTRE SEÑALES FNIRS
# ============================================

def analyze_fnirs_correlations(subject_id, fnirs_data):
    """
    Analiza correlaciones entre diferentes tipos de señales fNIRS
    """
    if len(fnirs_data) < 2:
        return

    # Crear DataFrame con todas las señales
    df_signals = pd.DataFrame()
    for signal_type, values in fnirs_data.items():
        df_signals[signal_type] = values[:min(1000, len(values))]  # Primeros 1000 puntos

    # Calcular matriz de correlación
    corr_matrix = df_signals.corr()

    # Visualizar
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0,
                vmin=-1, vmax=1, square=True)
    plt.title(f'Correlaciones entre señales fNIRS - {subject_id}')
    plt.tight_layout()
    plt.show()

    return corr_matrix

for subject in subjects[:1]:
    fnirs_data = load_all_fnirs_for_subject(subject)
    if fnirs_data:
        corr_matrix = analyze_fnirs_correlations(subject, fnirs_data)

In [ ]:
# ============================================
# 7. PREPROCESAMIENTO PARA MACHINE LEARNING
# ============================================

# Preparar datos para ML (como no tenemos variable objetivo explícita,
# usaremos el género como variable a predecir como ejemplo)
# En un caso real, aquí tendrías las calificaciones o métricas de rendimiento

# Separar características
feature_cols = [col for col in df_multimodal.columns
                if col not in ['subject_id', 'gender']]
X = df_multimodal[feature_cols].copy()
y = df_multimodal['gender']  # Usamos género como ejemplo

print(f"\n{'='*50}")
print("PREPROCESAMIENTO PARA MACHINE LEARNING")
print('='*50)
print(f"Características: {len(feature_cols)}")
print(f"Target: {y.value_counts().to_dict()}")

# Manejar valores NaN
print("\nValores NaN por columna:")
nan_counts = X.isna().sum()
print(nan_counts[nan_counts > 0])

# Imputar NaN con la media
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(X)
X_imputed = pd.DataFrame(X_imputed, columns=feature_cols)

# Escalar características
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols)

print(f"\nDataset procesado: {X_scaled.shape}")
print(f"NaN después de imputación: {X_scaled.isna().sum().sum()}")

In [ ]:
# ============================================
# 8. ANÁLISIS DE COMPONENTES PRINCIPALES (PCA)
# ============================================

# Aplicar PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Varianza explicada
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.bar(range(1, len(pca.explained_variance_ratio_) + 1),
        pca.explained_variance_ratio_, alpha=0.7)
plt.xlabel('Componente Principal')
plt.ylabel('Varianza Explicada')
plt.title('Varianza Explicada por Componente')

plt.subplot(1, 2, 2)
plt.plot(range(1, len(pca.explained_variance_ratio_) + 1),
         np.cumsum(pca.explained_variance_ratio_), 'bo-')
plt.xlabel('Número de Componentes')
plt.ylabel('Varianza Explicada Acumulada')
plt.title('Varianza Explicada Acumulada')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nVarianza explicada por los primeros 2 componentes: {pca.explained_variance_ratio_[:2].sum():.1%}")
print(f"Varianza explicada por los primeros 5 componentes: {pca.explained_variance_ratio_[:5].sum():.1%}")

# Visualizar PCA con colores por género
plt.figure(figsize=(10, 8))
colors = {'F': 'red', 'M': 'blue'}
for gender in ['F', 'M']:
    mask = y == gender
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=colors[gender], label=f'Género {gender}', alpha=0.7, s=100)

# Añadir etiquetas de sujetos
for i, subject in enumerate(df_multimodal['subject_id']):
    plt.annotate(subject, (X_pca[i, 0], X_pca[i, 1]),
                fontsize=9, ha='center')

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.title('Visualización PCA - Diferencias por Género')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# 9. ANÁLISIS DE CARACTERÍSTICAS MÁS IMPORTANTES
# ============================================

# Usar Random Forest para identificar características importantes
rf_temp = RandomForestClassifier(n_estimators=100, random_state=42)
rf_temp.fit(X_scaled, y)

importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_temp.feature_importances_
}).sort_values('importance', ascending=False)

# Visualizar top 15 características
plt.figure(figsize=(12, 8))
top_15 = importances.head(15)
plt.barh(range(len(top_15)), top_15['importance'].values, tick_label=top_15['feature'].values)
plt.xlabel('Importancia')
plt.title('Top 15 Características más Importantes (para clasificación de género)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 10 características más importantes:")
for i, row in importances.head(10).iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

In [ ]:
# ============================================
# 10. ANÁLISIS DE DIFERENCIAS POR GÉNERO
# ============================================

def analyze_gender_differences(X_scaled, y, feature_cols):
    """
    Analiza diferencias estadísticas entre géneros
    """
    results = []

    X_f = X_scaled[y == 'F']
    X_m = X_scaled[y == 'M']

    for i, feature in enumerate(feature_cols):
        f_values = X_f.iloc[:, i].values
        m_values = X_m.iloc[:, i].values

        # Calcular estadísticas
        f_mean = np.mean(f_values)
        m_mean = np.mean(m_values)

        # Prueba t (asumiendo varianzas iguales)
        from scipy.stats import ttest_ind
        t_stat, p_value = ttest_ind(f_values, m_values)

        results.append({
            'feature': feature,
            'f_mean': f_mean,
            'm_mean': m_mean,
            'difference': f_mean - m_mean,
            't_stat': t_stat,
            'p_value': p_value,
            'significant': p_value < 0.05
        })

    return pd.DataFrame(results)

gender_diff = analyze_gender_differences(X_scaled, y, feature_cols)

# Mostrar características con diferencias significativas
sig_diff = gender_diff[gender_diff['significant']].sort_values('p_value')
print("\nCaracterísticas con diferencias significativas entre géneros (p < 0.05):")
if len(sig_diff) > 0:
    for i, row in sig_diff.iterrows():
        direction = "mayor" if row['f_mean'] > row['m_mean'] else "menor"
        print(f"  {row['feature']}: p={row['p_value']:.4f}, "
              f"F={row['f_mean']:.2f}, M={row['m_mean']:.2f} "
              f"({direction} en F)")
else:
    print("  Ninguna característica muestra diferencias significativas")

# Visualizar top diferencias
plt.figure(figsize=(12, 6))
top_diff = gender_diff.nlargest(10, 'difference').sort_values('difference')
colors = ['red' if d > 0 else 'blue' for d in top_diff['difference']]
plt.barh(range(len(top_diff)), top_diff['difference'], color=colors, tick_label=top_diff['feature'])
plt.xlabel('Diferencia (F - M)')
plt.title('Top 10 Diferencias entre Géneros')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# 11. CLASIFICACIÓN DE GÉNERO (ejemplo)
# ============================================

print("\n" + "="*50)
print("CLASIFICACIÓN DE GÉNERO (ejemplo)")
print("="*50)

# Usar Leave-One-Out Cross-Validation (adecuado para dataset pequeño)
loo = LeaveOneOut()
y_true = []
y_pred = []

for train_idx, test_idx in loo.split(X_scaled):
    X_train, X_test = X_scaled.iloc[train_idx], X_scaled.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Entrenar clasificador
    clf = RandomForestClassifier(n_estimators=50, random_state=42)
    clf.fit(X_train, y_train)

    # Predecir
    y_pred_test = clf.predict(X_test)

    y_true.extend(y_test)
    y_pred.extend(y_pred_test)

# Calcular accuracy
accuracy = accuracy_score(y_true, y_pred)
print(f"Accuracy (Leave-One-Out): {accuracy:.2%}")
print(f"Clasificaciones correctas: {sum(np.array(y_true) == np.array(y_pred))}/{len(y_true)}")

In [ ]:
# ============================================
# 12. GUARDAR RESULTADOS
# ============================================

def save_results(df_multimodal, importances, gender_diff, X_scaled, feature_cols):
    """
    Guarda los resultados en archivos CSV
    """
    from datetime import datetime

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Guardar dataset completo
    df_multimodal.to_csv(f"multimodal_dataset_{timestamp}.csv", index=False)
    print(f"\nDataset guardado: multimodal_dataset_{timestamp}.csv")

    # Guardar importancias
    importances.to_csv(f"feature_importances_{timestamp}.csv", index=False)
    print(f"Importancias guardadas: feature_importances_{timestamp}.csv")

    # Guardar diferencias por género
    gender_diff.to_csv(f"gender_differences_{timestamp}.csv", index=False)
    print(f"Diferencias por género guardadas: gender_differences_{timestamp}.csv")

    # Guardar datos escalados
    X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)
    X_scaled_df['subject_id'] = df_multimodal['subject_id'].values
    X_scaled_df['gender'] = df_multimodal['gender'].values
    X_scaled_df.to_csv(f"scaled_features_{timestamp}.csv", index=False)
    print(f"Características escaladas guardadas: scaled_features_{timestamp}.csv")

# Guardar resultados
save_results(df_multimodal, importances, gender_diff, X_scaled, feature_cols)

In [ ]:
# ============================================
# 13. RESUMEN FINAL
# ============================================

print("\n" + "="*50)
print("RESUMEN DEL ANÁLISIS MULTIMODAL")
print("="*50)
print(f"""
ANÁLISIS COMPLETADO:

Dataset:
- Sujetos analizados: {len(df_multimodal)}
- Géneros: {df_multimodal['gender'].value_counts().to_dict()}
- Características totales: {len(feature_cols)}
- Tipos de señales fNIRS: ohb, dohb, tothb, O2

Resultados principales:
1. PCA: {pca.explained_variance_ratio_[:2].sum():.1%} de varianza en 2 componentes
2. Clasificación de género: {accuracy:.2%} accuracy
3. Características más importantes: {importances.iloc[0]['feature']} ({importances.iloc[0]['importance']:.4f})
4. Diferencias significativas por género: {len(sig_diff)} características

Archivos guardados:
- Dataset completo
- Importancias de características
- Diferencias por género
- Características escaladas
""")

# Visualización final: mapa de calor de correlaciones de características importantes
plt.figure(figsize=(14, 12))
top_features = importances.head(20)['feature'].tolist()
X_top = X_scaled[top_features]
corr_top = X_top.corr()

sns.heatmap(corr_top, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, cbar_kws={"shrink": 0.8})
plt.title('Correlaciones entre las 20 características más importantes')
plt.tight_layout()
plt.show()

ImportError: cannot import name 'trapz' from 'scipy.integrate' (/usr/local/lib/python3.12/dist-packages/scipy/integrate/__init__.py)